In [1]:
! pip install langchain_community tikroken lanchain-openai langchainhub chromeadb langchain

ERROR: Could not find a version that satisfies the requirement tikroken (from versions: none)
ERROR: No matching distribution found for tikroken


In [1]:
import os



In [6]:
! pip install chromadb


In [6]:
import os
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")



In [7]:
os.environ["LANGSMITH_TRACING_V2"]='true'
os.environ['LANGSMITH_ENDPOINT']='https://api.smith.langchain.com'

from dotenv import load_dotenv
load_dotenv()


from langchain_anthropic import ChatAnthropic





In [8]:
import bs4
from langsmith import Client
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
# one-time install
# uv add sentence-transformers

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_anthropic import ChatAnthropic





#### INDEXING ####

# Load Documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs = loader.load()
# Split
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

splits = text_splitter.split_documents(docs)
# Embed
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding)

retriever = vectorstore.as_retriever()
#### RETRIEVAL and GENERATION ####

# Prompt
prompt = Client().pull_prompt("rlm/rag-prompt")

# LLM

llm = ChatAnthropic(
    model="claude-3-haiku-20240307",
    temperature=0
)
# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
    # Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
# Question

rag_chain.invoke("What is Task Decomposition?")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9080.49it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


'I don\'t know the answer to the question "What is Task Decomposition?". The provided context does not directly define or explain what task decomposition is. The context discusses different approaches to task decomposition, such as using language models with prompting, task-specific instructions, or human inputs, as well as an approach involving external classical planners. However, it does not provide a clear, concise definition of the term "task decomposition" itself.'